In [ ]:
import pandas as pd


It seems the file path `../data/mtsamples/mtsamples.csv` is incorrect. Let's list the files available in the environment to locate the correct path for your CSV file.

In [ ]:
# List all files recursively in the current directory
!ls -R

.:
sample_data

./sample_data:
anscombe.json		      mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md


In [ ]:
mtsamples = pd.read_csv("../data/mtsamples/mtsamples.csv")

print("Number of records:", len(mtsamples))
print("\nColumns:")
for column in mtsamples.columns:
    print("-", column)

FileNotFoundError: [Errno 2] No such file or directory: '../data/mtsamples/mtsamples.csv'

In [ ]:
# Install necessary libraries for web scraping
!pip install requests beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

Below is a function to scrape details from a *single sample page*. The HTML selectors (`find('tag', class_='...')`) are **hypothetical** and will need to be adjusted based on the actual HTML structure of `mtsamples.com`. You can use your browser's developer tools (right-click -> Inspect) on an actual sample page to find the correct tags, classes, or IDs for each piece of information.

In [ ]:
def scrape_sample_details(sample_url):
    """Scrapes description, medical specialty, sample name, transcription, and keywords from a single sample page."""
    # Add a User-Agent header to mimic a web browser and avoid 403 Forbidden errors
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    try:
        response = requests.get(sample_url, headers=headers)
        response.raise_for_status()  # Raise an exception for HTTP errors
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {sample_url}: {e}")
        return None

    soup = BeautifulSoup(response.content, 'html.parser')

    # --- Hypothetical Selectors (MUST BE ADJUSTED) ---
    # You need to inspect the HTML of an actual sample page (e.g., https://www.mtsamples.com/site/pages/sample.asp?id=123 if such a page exists)
    # and find the correct CSS selectors (tags, classes, ids) for each piece of data.

    medical_specialty = "N/A"
    sample_name = "N/A"
    description = "N/A"
    transcription = "N/A"
    keywords = "N/A"

    # Example: Trying to find elements by common tags/classes. This is a starting point.
    # For 'medical_specialty', it might be in a header or a specific metadata field.
    # For 'sample_name', often an <h1> tag.
    # For 'description', a <p> tag following a 'Description' heading.
    # For 'transcription', a large text block in a specific div.
    # For 'keywords', often in a meta tag or a specific list.

    # Placeholder for 'medical_specialty'
    specialty_tag = soup.find('h3', string=lambda s: s and 'Medical Specialty:' in s) # Example: if it's in an h3
    if specialty_tag:
        medical_specialty = specialty_tag.get_text(strip=True).replace('Medical Specialty:', '').strip()

    # Placeholder for 'sample_name'
    name_tag = soup.find('h1', class_='sample-title') # Example: if the title is in an h1 with class 'sample-title'
    if name_tag:
        sample_name = name_tag.get_text(strip=True)
    elif soup.find('title'): # Fallback to page title
        sample_name = soup.find('title').get_text(strip=True)

    # Placeholder for 'description'
    desc_div = soup.find('div', class_='description-section') # Example: if description is in a div
    if desc_div:
        description = desc_div.get_text(separator=' ', strip=True)

    # Placeholder for 'transcription'
    trans_div = soup.find('div', class_='transcription-text') # Example: if transcription is in a div
    if trans_div:
        transcription = trans_div.get_text(separator=' ', strip=True)

    # Placeholder for 'keywords'
    keywords_meta = soup.find('meta', {'name': 'keywords'}) # Example: if keywords are in a meta tag
    if keywords_meta and 'content' in keywords_meta.attrs:
        keywords = keywords_meta['content']
    else:
        keywords_tag = soup.find('p', class_='keywords') # Example: if keywords are in a p tag
        if keywords_tag:
            keywords = keywords_tag.get_text(strip=True)


    data = {
        'description': description,
        'medical_specialty': medical_specialty,
        'sample_name': sample_name,
        'transcription': transcription,
        'keywords': keywords
    }
    return data

Now, let's demonstrate the `scrape_sample_details` function with a **hypothetical** sample URL. You will need to replace `actual_sample_url` with a real URL from `mtsamples.com` that links to an individual sample (e.g., `https://www.mtsamples.com/site/pages/sample.asp?id=1`).

In [ ]:
# IMPORTANT: Replace this with an actual sample URL from mtsamples.com
# You might need to first navigate to a category page, then pick a sample from there.
actual_sample_url = "https://www.mtsamples.com/site/pages/sample.asp?id=1" # Placeholder

print(f"Attempting to scrape details from: {actual_sample_url}")
sample_details = scrape_sample_details(actual_sample_url)

if sample_details:
    df_sample = pd.DataFrame([sample_details])
    print("\nScraped Sample Details:")
    display(df_sample)
else:
    print("Failed to scrape sample details. Please check the URL and refine the HTML selectors in the `scrape_sample_details` function.")

Attempting to scrape details from: https://www.mtsamples.com/site/pages/sample.asp?id=1
Error fetching https://www.mtsamples.com/site/pages/sample.asp?id=1: 403 Client Error: Forbidden for url: https://www.mtsamples.com/site/pages/sample.asp?id=1
Failed to scrape sample details. Please check the URL and refine the HTML selectors in the `scrape_sample_details` function.


**Next Steps for Comprehensive Scraping:**

1.  **Find Actual Sample URLs:** The `actual_sample_url` used above is a placeholder. To get a full dataset, you'll need to:
    *   Visit the main page (`https://www.mtsamples.com/`).
    *   Identify and extract links to medical specialties (e.g., "Cardiology", "Neurology").
    *   Navigate to each specialty page and extract links to individual medical transcription samples.
2.  **Refine HTML Selectors:** Carefully inspect the HTML of an actual sample page using your browser's developer tools. Update the `soup.find()` calls within the `scrape_sample_details` function to use the correct HTML tags, classes, and IDs for `description`, `medical_specialty`, `sample_name`, `transcription`, and `keywords`.
3.  **Iterate and Collect Data:** Once you have a working `scrape_sample_details` function with correct selectors and a method to obtain all sample URLs, you can loop through all desired samples, call this function for each, and append the results to a list. Finally, convert the list of dictionaries into a single pandas DataFrame.
4. **Save the Data:** After collecting all data, you can save it to a CSV file using `df.to_csv('mtsamples.csv', index=False)`.